# MNIST N-Point Correlators & Polyspectra Analysis

This notebook computes N=2,3 connected correlators (cumulants) on MNIST images and transforms them to Fourier/wavelet polyspectra.

## Overview
- **2-point correlator**: $C_2(r) = \langle I(x) I(x+r) \rangle - \langle I \rangle^2$ → Power spectrum $P(k)$
- **3-point correlator**: $C_3(r_1, r_2)$ (cumulant) → Bispectrum $B(k_1, k_2, k_3)$
- Both Fourier and wavelet transforms
- Per-digit class analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.datasets import fetch_openml

from src.config import CorrelatorConfig, SamplingConfig, PolyspectraConfig, VisualizationConfig
from src.correlators import compute_connected_2point, compute_connected_3point, compute_per_digit_correlators
from src.polyspectra import compute_power_spectrum, compute_bispectrum, compute_wavelet_spectrum, fit_power_law
from src.visualization import (
    plot_power_spectrum, plot_power_spectrum_comparison,
    plot_bispectrum_equilateral, plot_bispectrum_2d,
    plot_wavelet_spectra, plot_digit_comparison_heatmap,
    create_summary_figure
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load MNIST Data

In [ ]:
# Load MNIST
print("Loading MNIST...")
mnist = fetch_openml('mnist_784', version=1, parser='auto')
images = mnist.data.values.reshape(-1, 28, 28).astype(np.float32)
labels = mnist.target.astype(int).values

# Use subset for faster computation (remove this line for full analysis)
n_samples = 10000
indices = np.random.RandomState(42).choice(len(images), n_samples, replace=False)
images = images[indices]
labels = labels[indices]

print(f"Loaded {len(images)} images")
print(f"Image shape: {images.shape[1:]}")
print(f"Labels: {np.unique(labels)}")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit in range(10):
    idx = np.where(labels == digit)[0][0]
    ax = axes[digit // 5, digit % 5]
    ax.imshow(images[idx], cmap='gray')
    ax.set_title(f'Digit {digit}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Configure Analysis

In [ ]:
# Create configurations
corr_config = CorrelatorConfig(
    n_points=3,
    max_separation=14,
    n_samples_per_config=500,  # Increase for better statistics
    n_configurations=300,       # Increase for more coverage
    translation_invariant=True
)

samp_config = SamplingConfig(
    separation_grid=[1, 2, 4, 7, 10, 14],
    n_angles=8,
    emphasize_geometries=['equilateral', 'squeezed', 'collinear'],
    geometry_oversample=2.0
)

poly_config = PolyspectraConfig(
    use_fourier=True,
    use_wavelet=True,
    wavelet_family='db4',
    n_wavelet_scales=4
)

vis_config = VisualizationConfig(
    per_digit=True,
    figure_dpi=150
)

# Create output directory
results_dir = Path.cwd().parent / 'results'
figures_dir = results_dir / 'figures'
data_dir = results_dir / 'data'
figures_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

print("Configurations created!")

## 3. Compute Overall Correlators

In [ ]:
# Compute C2
print("\nComputing C2...")
C2_result = compute_connected_2point(images, corr_config, samp_config, verbose=True)
print(f"C2 computed: {len(C2_result['C2'])} configurations")

In [ ]:
# Compute C3
print("\nComputing C3...")
C3_result = compute_connected_3point(images, corr_config, samp_config, verbose=True)
print(f"C3 computed: {len(C3_result['C3'])} configurations")

In [ ]:
# Save correlator data
np.savez(
    data_dir / 'correlators_overall.npz',
    **C2_result,
    **{f'C3_{k}': v for k, v in C3_result.items()}
)
print("Correlators saved!")

## 4. Compute Fourier Polyspectra

In [ ]:
# Power spectrum
print("\nComputing power spectrum...")
P_data = compute_power_spectrum(C2_result, poly_config)
fit_data = fit_power_law(P_data['k_values'], P_data['P_k_radial'])
print(f"Power-law exponent: α = {fit_data['exponent']:.3f} (R² = {fit_data['r_squared']:.3f})")

# Plot
fig = plot_power_spectrum(P_data, fit_data, config=vis_config)
fig.savefig(figures_dir / 'power_spectrum_overall.png', bbox_inches='tight')
plt.show()

In [ ]:
# Bispectrum
print("\nComputing bispectrum...")
B_data = compute_bispectrum(C3_result, poly_config)
print(f"Bispectrum components: {list(B_data.keys())}")

# Plot bispectrum panels
fig_eq = plot_bispectrum_equilateral(B_data, config=vis_config)
fig_eq.savefig(figures_dir / 'bispectrum_equilateral_overall.png', bbox_inches='tight')
plt.show()

fig_2d = plot_bispectrum_2d(B_data, config=vis_config)
fig_2d.savefig(figures_dir / 'bispectrum_2d_overall.png', bbox_inches='tight')
plt.show()

## 5. Compute Wavelet Polyspectra

In [ ]:
# Wavelet spectrum
print("\nComputing wavelet spectrum...")
wavelet_data = compute_wavelet_spectrum(C2_result, poly_config)

fig = plot_wavelet_spectra(wavelet_data, config=vis_config)
fig.savefig(figures_dir / 'wavelet_spectrum_overall.png', bbox_inches='tight')
plt.show()

## 6. Summary Figure

In [ ]:
# Create comprehensive summary figure
fig = create_summary_figure(
    P_data, B_data, wavelet_data, fit_data,
    config=vis_config
)
fig.savefig(figures_dir / 'summary_overall.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Per-Digit Analysis

In [ ]:
# Compute correlators for each digit
# NOTE: This takes ~10x longer than overall analysis
# Reduce n_samples_per_config and n_configurations for faster computation

corr_config_fast = CorrelatorConfig(
    n_points=3,
    max_separation=14,
    n_samples_per_config=200,  # Reduced for speed
    n_configurations=150,       # Reduced for speed
    translation_invariant=True
)

print("\nComputing per-digit correlators...")
per_digit_correlators = compute_per_digit_correlators(
    images, labels, corr_config_fast, samp_config, verbose=True
)

In [ ]:
# Compute polyspectra for each digit
per_digit_P = {}
per_digit_B = {}
per_digit_wavelet = {}
per_digit_fits = {}

for digit in range(10):
    print(f"\nProcessing digit {digit}...")
    C2_digit = per_digit_correlators[digit]['C2']
    C3_digit = per_digit_correlators[digit]['C3']
    
    # Power spectrum
    P_data = compute_power_spectrum(C2_digit, poly_config)
    fit_data = fit_power_law(P_data['k_values'], P_data['P_k_radial'])
    per_digit_P[digit] = P_data
    per_digit_fits[digit] = fit_data
    
    # Bispectrum
    B_data = compute_bispectrum(C3_digit, poly_config)
    per_digit_B[digit] = B_data
    
    # Wavelet
    wavelet_data = compute_wavelet_spectrum(C2_digit, poly_config)
    per_digit_wavelet[digit] = wavelet_data
    
    print(f"  Power-law exponent: α = {fit_data['exponent']:.3f}")

In [ ]:
# Plot power spectrum comparison
fig = plot_power_spectrum_comparison(per_digit_P, vis_config)
fig.savefig(figures_dir / 'power_spectrum_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Create summary statistics for digit comparison
all_digits_stats = {}

for digit in range(10):
    all_digits_stats[digit] = {
        'exponent': per_digit_fits[digit]['exponent'],
        'bispectrum_mean': np.nanmean(per_digit_B[digit].get('B_equilateral', {}).get('values', [0])),
        'wavelet_scale_mean': np.mean(per_digit_wavelet[digit]['P_scale'])
    }

# Plot comparison heatmap
fig = plot_digit_comparison_heatmap(all_digits_stats, vis_config)
fig.savefig(figures_dir / 'digit_comparison_heatmap.png', bbox_inches='tight')
plt.show()

## 8. Statistical Summary

In [ ]:
# Print summary table
print("\n" + "="*60)
print("SUMMARY STATISTICS BY DIGIT")
print("="*60)
print(f"{'Digit':>6} {'Power-law α':>12} {'Bispectrum':>12} {'Wavelet':>12}")
print("-"*60)

for digit in range(10):
    stats = all_digits_stats[digit]
    print(f"{digit:>6} {stats['exponent']:>12.3f} {stats['bispectrum_mean']:>12.3e} {stats['wavelet_scale_mean']:>12.3e}")

print("="*60)

In [ ]:
# Save all results
np.savez(
    data_dir / 'polyspectra_all.npz',
    per_digit_stats=all_digits_stats,
    overall_fit=fit_data
)
print("\nAll results saved to:", data_dir)

## 9. Interactive Exploration (Optional)

Uncomment and run the cells below for interactive plotly visualizations.

In [ ]:
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# # Interactive power spectrum comparison
# fig = make_subplots(rows=1, cols=1)

# for digit in range(10):
#     P_data = per_digit_P[digit]
#     k = P_data['k_values']
#     P = P_data['P_k_radial']
#     valid = (k > 0) & (P > 0)
#     
#     fig.add_trace(go.Scatter(
#         x=k[valid], y=P[valid],
#         mode='lines+markers',
#         name=f'Digit {digit}'
#     ))

# fig.update_xaxes(type='log', title='k (1/pixels)')
# fig.update_yaxes(type='log', title='P(k)')
# fig.update_layout(title='Interactive Power Spectrum Comparison', height=600)
# fig.show()

## Summary

This notebook computed:
- 2-point and 3-point connected correlators on MNIST
- Fourier polyspectra (power spectrum and bispectrum)
- Wavelet polyspectra for scale-dependent analysis
- Per-digit class comparisons

**Key findings:**
- Power spectrum typically shows $P(k) \sim k^{\alpha}$ with $\alpha \approx -2$ (scale invariance)
- Non-zero bispectrum indicates non-Gaussian structure
- Different digit classes exhibit distinct higher-order statistical signatures

**Next steps:**
- Apply to learned representations (hidden layers of trained networks)
- Extend to N=4 (trispectrum) for richer structure
- Compare to other datasets (CIFAR, natural images)
- Connect to integrated information measures